# VAE-BM vs. published baselines (FASTopic, GloCOM) - Colab runner

Reproduces each baseline paper's OWN protocol (dataset artifact, preprocessing, K, metrics) and evaluates VAE-BM under the exact same conditions. See the repo README and `docs/methodological_notes.md` for the full rationale - this notebook is a thin runner, not where the protocol decisions live.

Steps: clone -> install -> GPU check -> verify protocol -> run baseline -> run VAE-BM -> compare -> final table.

## 1. Clone

In [ ]:
!git clone <THIS_REPO_URL> vaebm-baselines-comparision
%cd vaebm-baselines-comparision

## 2. Install

In [ ]:
!pip install -e ".[all]" -q

## 3. GPU check

In [ ]:
import torch, tensorflow as tf
print('torch CUDA available:', torch.cuda.is_available())
print('tensorflow GPUs:', tf.config.list_physical_devices('GPU'))
print('If both are empty/False, everything below still runs on CPU (slower) - both protocols were verified to work on CPU during this project\'s own research pass.')

## 4. FASTopic protocol - NYT

Dataset fetched automatically by `topmost.download_dataset('NYT', ...)` inside the scripts below - no manual download step needed.

In [ ]:
!python scripts/verify_protocol.py --baseline fastopic

In [ ]:
!python scripts/run_baseline.py --baseline fastopic --dataset nyt --seed 42

In [ ]:
!python scripts/run_vaebm.py --protocol fastopic --dataset nyt --seed 42

In [ ]:
!python scripts/compare.py --baseline fastopic --dataset nyt

## 5. GloCOM protocol - SearchSnippets

Dataset fetched automatically from GloCOM's own official repo (precomputed `bow.npz`/`global_bow.npz`/`global_maps.txt`/`vocab.txt` - see `docs/methodological_notes.md` for why this is the most faithful path).

In [ ]:
!python scripts/verify_protocol.py --baseline glocom

In [ ]:
!python scripts/run_baseline.py --baseline glocom --dataset search_snippets --seed 42

In [ ]:
!python scripts/run_vaebm.py --protocol glocom --dataset search_snippets --seed 42

In [ ]:
!python scripts/compare.py --baseline glocom --dataset search_snippets

## 6. Final table

Published vs. reproduced-baseline vs. VAE-BM, both protocols, side by side. A gap between `published` and `reproduced` is expected (different hardware/package versions/exact reference corpus - see each protocol's docstring) and must be visible before any claim that VAE-BM outperforms a baseline.

In [ ]:
import pandas as pd

for baseline, dataset in [("fastopic", "nyt"), ("glocom", "search_snippets")]:
    print(f"\n=== {baseline} / {dataset} ===")
    display(pd.read_csv(f"results/{baseline}/comparison.csv"))

## 7. This is a smoke test

Both protocols above ran with reduced epochs (20 instead of the papers' 200) and a single seed, per this project's own instructions ("test only small experiments"). Pass `--full` to any `run_baseline.py`/`run_vaebm.py`/`verify_protocol.py` call to use paper-scale settings once the smoke test looks sane.